In [1]:
import langchain_openai

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [3]:
import json

In [4]:
import webbrowser

In [5]:
import speech_recognition as sr

In [6]:
from gtts import gTTS

In [7]:
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.11.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [8]:
from io import BytesIO

In [9]:
recognizer = sr.Recognizer()

In [10]:
def recognize_speech():
    text=""
    with sr.Microphone() as source:
        print("Konuşmaya başlayabilirsiniz...")
        audio = recognizer.listen(source)
        
        try:
            text = recognizer.recognize_google(audio, language='tr-TR')
        except sr.UnknownValueError:
            print("Üzgünüm, ne dediğinizi anlayamadım.")
        except sr.RequestError as e:
            print("Servise erişilemedi; hata: {0}".format(e))
    return(text)

In [11]:
def text_to_speech(text, language='en'):
    try:
        # Metni sese dönüştür ve bellekte sakla
        tts = gTTS(text=text, lang=language, slow=False)
        audio_data = BytesIO()
        tts.write_to_fp(audio_data)
        audio_data.seek(0)

        # Pygame ile sesi çal
        pygame.mixer.init()
        pygame.mixer.music.load(audio_data, "mp3")
        pygame.mixer.music.play()

        # Sesin bitmesini bekle
        while pygame.mixer.music.get_busy():
            continue

    except Exception as e:
        print(f"Bir hata oluştu: {e}")

In [12]:
from langchain_openai import ChatOpenAI

In [13]:
functions = [
    {
        "name": "open_music",
        "description": "istediğin müzigi calar",
        "parameters": {
            "type": "object",
            "properties": {
                "parcaadi": {
                    "type": "string",
                    "description": "Çalınacak müziğin ismi"
                }
            },
            "required": ["parcaadi"]
        }
    },
    {
        "name": "open_newspaper",
        "description": "istediğin gazetenin internet sayfasını açar",
        "parameters": {
            "type": "object",
            "properties": {
                "gazeteadi": {
                    "type": "string",
                    "description": "internet sayfası açılacak gazetenin adı"
                }
            },
            "required": ["gazeteadi"]
        }
    }
]

In [14]:
chatModel = ChatOpenAI(
    model="gpt-4o-mini",
    model_kwargs={"functions": functions}
)

In [15]:
def open_music(parcaadi):
    global input_history
    print("parcaadi:"+parcaadi)
    input_history=""
    if parcaadi=='hercai' or parcaadi=='Hercai':        
        os.startfile(r"hercai.mp4")

In [16]:
def open_newspaper(gazeteadi):
    global input_history
    print("gazeteadi:"+gazeteadi)
    input_history=".."
    if gazeteadi=='milliyet' or gazeteadi=='Milliyet':        
        webbrowser.open("https://www.milliyet.com.tr/")
    if gazeteadi=='sabah' or gazeteadi=='Sabah':
        webbrowser.open("https://www.sabah.com.tr/")

In [19]:
 input_history=""

In [20]:
while True:
    # Kullanıcıdan girdi al
    user_input = recognize_speech()
    input_history=input_history+'\n\n'+user_input

    
    # Eğer kullanıcı "exit" yazarsa döngüden çık
    if user_input.lower() == "exit":
        print("Çıkış yapılıyor...")
        break
    messages = [
    ("system", "Sohbet etmeyi seven neşeli bir arkadaşsın kullanıcı ile sohbet et eğer bir isteği varsa tanımlı function lardan bu isteği yerine getir. Emoji kullanma"),
    ("human",input_history),
    ("human",user_input)
    ]

    response = chatModel.invoke(messages)

    try:
        if(response.additional_kwargs['function_call']['name']=='open_music'):
            arguments = json.loads(response.additional_kwargs['function_call']['arguments'])
            open_music(arguments['parcaadi'])
        if(response.additional_kwargs['function_call']['name']=='open_newspaper'):
            arguments = json.loads(response.additional_kwargs['function_call']['arguments'])
            open_newspaper(arguments['gazeteadi'])
    except KeyError:
        text_to_speech(response.content, language='tr')


Konuşmaya başlayabilirsiniz...
Konuşmaya başlayabilirsiniz...
Konuşmaya başlayabilirsiniz...
Konuşmaya başlayabilirsiniz...
gazeteadi:sabah
Konuşmaya başlayabilirsiniz...
Konuşmaya başlayabilirsiniz...
parcaadi:hercai
Konuşmaya başlayabilirsiniz...


KeyboardInterrupt: 